# 04 - CA-CWGAN with the VAE density filter (E6)

Re-samples the generator trained in notebook 03 (no training here) and replaces
the k-NN vote with a per-class VAE likelihood test. GPU. Run once per dataset.
See `PIPELINE.md`, step 4.

In [ ]:
# Colab: upload df_train_<dataset>.csv and ckpt_ecwgan_E12345_<dataset>_..._seed42.pt (notebook 03).
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    files.upload()

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.mixture import BayesianGaussianMixture

try:
    display
except NameError:
    display = print

## Config

In [ ]:
DATASET  = "diabetes"     # diabetes | yeast | ecoli | pageblocks
DATA_DIR = "."
CHECKPOINT_DIR = "."      # folder holding notebook 03's checkpoint

SETTING = dict(updates=10000, batch=250, lr=2e-4, noise_dim=128)   # must match notebook 03

SEED       = 42
N_MODES    = 10           # E2, as in notebook 03
TAU        = 0.0          # E3, as in notebook 03
GUMBEL_TAU = 0.2          # E4a, as in notebook 03
PAC        = 10           # E4b, as in notebook 03
MAX_ROUNDS = 8
MAX_DISCRETE_LEVELS = 12
GEN_CHUNK  = 131072
ACTIVE     = "12345"

# E6
VAE_LATENT   = 8
VAE_HIDDEN   = 64
VAE_EPOCHS   = 200
VAE_BATCH    = 128
VAE_LR       = 1e-3
VAE_MARGIN   = 0.0        # keep when the own-class score beats the best rival by more than this
MIN_VAE_ROWS = 30         # classes with fewer real rows get no VAE (see PIPELINE.md)

# Rows requested per round are sized from the acceptance rate observed so far.
MIN_DRAW     = 256
MIN_ACCEPT   = 0.02
MAX_OVERDRAW = 8
INITIAL_RATE = 0.5        # assumed acceptance rate for the first round

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. In Colab: Runtime > Change runtime type > T4 GPU.")
DEVICE = "cuda"

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, f"df_train_{DATASET}.csv"))
TARGET_COL = df_train.columns[-1]
FEATURES = [c for c in df_train.columns if c != TARGET_COL]
COUNTS = df_train[TARGET_COL].value_counts().sort_index()
N_MAX  = int(COUNTS.max())

DISCRETE_COLS, CONTINUOUS_COLS, LEVELS = [], [], {}
for col in FEATURES:
    lv = np.unique(df_train[col].to_numpy(dtype=float))
    if len(lv) <= MAX_DISCRETE_LEVELS:
        DISCRETE_COLS.append(col); LEVELS[col] = np.sort(lv)
    else:
        CONTINUOUS_COLS.append(col)

print("train:", df_train.shape, "| class counts:", COUNTS.tolist())

## CA-CWGAN generator (identical to notebook 03)

In [ ]:
class ColumnTransform:
    """Discrete -> one-hot. Continuous -> one-hot mode + offset inside that mode."""

    def fit(self, df):
        self.spans = [(c, "discrete", len(LEVELS[c])) for c in DISCRETE_COLS]
        self.gmm, self.active = {}, {}
        for c in CONTINUOUS_COLS:
            v = df[c].to_numpy(dtype=float).reshape(-1, 1)
            g = BayesianGaussianMixture(
                n_components=min(N_MODES, max(1, len(np.unique(v)))),
                weight_concentration_prior_type="dirichlet_process",
                weight_concentration_prior=1e-3, max_iter=100, random_state=SEED)
            g.fit(v)
            act = g.weights_ > 1e-3
            if not act.any():
                act[np.argmax(g.weights_)] = True
            self.gmm[c], self.active[c] = g, act
            self.spans.append((c, "vgm", 1 + int(act.sum())))
        self.dim = sum(w for _, _, w in self.spans)
        return self

    def inverse(self, M):
        out, pos = {}, 0
        for col, kind, width in self.spans:
            blk = M[:, pos:pos + width]; pos += width
            if kind == "discrete":
                out[col] = LEVELS[col][blk.argmax(axis=1)]
            else:
                g, act = self.gmm[col], self.active[col]
                mu = g.means_.reshape(-1)[act]
                sd = np.sqrt(g.covariances_.reshape(-1))[act]
                mode = blk[:, 1:].argmax(axis=1)
                out[col] = np.clip(np.clip(blk[:, 0], -1, 1) * 4 * sd[mode] + mu[mode], 0, 1)
        return pd.DataFrame(out)[FEATURES]

    def heads(self):
        plan, pos = [], 0
        for col, kind, width in self.spans:
            if kind == "discrete":
                plan.append(("softmax", pos, width))
            else:
                plan.append(("tanh", pos, 1)); plan.append(("softmax", pos + 1, width - 1))
            pos += width
        return plan


PACK = PAC


class Generator(nn.Module):
    def __init__(self, noise_dim, out_dim, cond_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim + cond_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, out_dim),
        )

    def forward(self, z, y):
        return self.net(torch.cat([z, y], dim=1))


def activate(raw, hard=False):
    parts = []
    for kind, start, width in CT.heads():
        blk = raw[:, start:start + width]
        if kind == "tanh":
            parts.append(torch.tanh(blk))
        else:
            parts.append(F.gumbel_softmax(blk, tau=GUMBEL_TAU, hard=hard, dim=1))
    return torch.cat(parts, dim=1)


def prepare(df):
    global CT
    CT = ColumnTransform().fit(df)
    classes = np.sort(df[TARGET_COL].unique())
    return {"ct": CT, "classes": classes, "cond_dim": len(classes),
            "eye": torch.eye(len(classes), device=DEVICE)}


def make_generate(G, ctx, cfg):
    classes, eye, ct = ctx["classes"], ctx["eye"], ctx["ct"]
    step = max(PACK, (GEN_CHUNK // PACK) * PACK)

    def generate(cls, n):
        idx = int(np.flatnonzero(classes == cls)[0])
        chunks, made = [], 0
        with torch.no_grad():
            while made < n:
                size = max(min(step, n - made), PACK)
                z = torch.randn(size, cfg["noise_dim"], device=DEVICE)
                cond = eye[torch.full((size,), idx, dtype=torch.long, device=DEVICE)]
                chunks.append(activate(G(z, cond), hard=True).cpu().numpy())
                made += size
        return ct.inverse(np.concatenate(chunks)[:n])

    return generate


def make_align(ref_df):
    """E5's per-class range alignment, kept in front of the E6 test."""
    ranges = {c: (g[FEATURES].min(), g[FEATURES].max()) for c, g in ref_df.groupby(TARGET_COL)}

    def align(df_part, cls):
        out = df_part.copy()
        lo, hi = ranges[cls]
        out[FEATURES] = out[FEATURES].clip(lo, hi, axis=1)
        for col in FEATURES:
            if col in LEVELS:
                lv = LEVELS[col]
                v = out[col].to_numpy(dtype=float)
                pos = np.clip(np.searchsorted(lv, v), 1, len(lv) - 1)
                left, right = lv[pos - 1], lv[pos]
                out[col] = np.where(np.abs(v - left) <= np.abs(v - right), left, right)
        return out

    return align


def rng_restore(s):
    random.setstate(s["python"])
    np.random.set_state(s["numpy"])
    torch.set_rng_state(s["torch"].cpu() if hasattr(s["torch"], "cpu") else s["torch"])
    if s.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([t.cpu() for t in s["cuda"]])

## E6 - One VAE per class

In [ ]:
class TinyVAE(nn.Module):
    def __init__(self, d_in, hidden=VAE_HIDDEN, latent=VAE_LATENT):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d_in, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU())
        self.mu = nn.Linear(hidden, latent)
        self.logvar = nn.Linear(hidden, latent)
        self.dec = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, d_in))

    def forward(self, x):
        h = self.enc(x)
        mu, logvar = self.mu(h), self.logvar(h).clamp(-8, 8)
        std = torch.exp(0.5 * logvar)
        z = mu + std * torch.randn_like(std)
        return self.dec(z), mu, logvar


def vae_elbo(model, x, reduce=True):
    """Negative ELBO per row (lower is better); unit-variance Gaussian decoder."""
    recon, mu, logvar = model(x)
    rec = ((recon - x) ** 2).sum(dim=1)
    kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1)
    out = rec + kl
    return out.mean() if reduce else out


def fit_class_vaes(df, seed=SEED):
    """One VAE per class with at least MIN_VAE_ROWS real rows."""
    models, skipped = {}, []
    for cls, g in df.groupby(TARGET_COL):
        if len(g) < MIN_VAE_ROWS:
            skipped.append(int(cls))
            continue
        torch.manual_seed(seed + int(cls))
        X = torch.tensor(g[FEATURES].to_numpy(dtype=np.float32), device=DEVICE)
        m = TinyVAE(X.shape[1]).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=VAE_LR)
        m.train()
        for _ in range(VAE_EPOCHS):
            idx = torch.randperm(len(X), device=DEVICE)
            for s in range(0, len(X), VAE_BATCH):
                xb = X[idx[s:s + VAE_BATCH]]
                if len(xb) < 2:
                    continue
                loss = vae_elbo(m, xb)
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        m.eval()
        models[int(cls)] = m
    print("VAE fitted for classes:", sorted(models), "| no VAE:", skipped)
    return models, skipped


@torch.no_grad()
def vae_keep(part, cls, models, chunk=65536):
    """True where the class-cls VAE explains the row better than every rival VAE."""
    if cls not in models:
        return np.ones(len(part), dtype=bool)
    X = part[FEATURES].to_numpy(dtype=np.float32)
    own, best_other = [], []
    for s in range(0, len(X), chunk):
        xb = torch.tensor(X[s:s + chunk], device=DEVICE)
        scores = {c: vae_elbo(m, xb, reduce=False) for c, m in models.items()}
        own.append(scores[cls].cpu().numpy())
        rivals = [v for c, v in scores.items() if c != cls]
        best_other.append(torch.stack(rivals).min(dim=0).values.cpu().numpy()
                          if rivals else np.full(len(xb), np.inf))
    return (np.concatenate(best_other) - np.concatenate(own)) > VAE_MARGIN

## Rebuild the pool with the E6 filter

In [ ]:
def build_pool_vae(G, ctx, cfg, ref_df, per_class, models):
    generate = make_generate(G, ctx, cfg)
    align = make_align(ref_df)

    def request(short, have, drawn):
        rate = (have / drawn) if drawn else INITIAL_RATE
        want = int(np.ceil(short / max(rate, MIN_ACCEPT)))
        return int(min(max(want, MIN_DRAW), MAX_OVERDRAW * short + MIN_DRAW))

    G.eval()
    parts, log = [], []
    for cls in ctx["classes"]:
        cls = int(cls)
        kept, have, drawn, rounds = [], 0, 0, 0
        while have < per_class and rounds < MAX_ROUNDS:
            want = request(per_class - have, have, drawn)
            part = generate(cls, want)
            part[TARGET_COL] = cls
            drawn += want
            part = align(part, cls)
            if len(part):
                part = part[vae_keep(part, cls, models)].reset_index(drop=True)
            kept.append(part)
            have += len(part)
            rounds += 1
        part = pd.concat(kept, ignore_index=True).iloc[:per_class].reset_index(drop=True)
        if len(part) < per_class:
            raise RuntimeError(f"class {cls}: only {len(part):,} of {per_class:,} rows survived")
        parts.append(part)
        log.append({"class": cls, "kept": len(part), "generated": drawn, "rounds": rounds,
                    "survival %": round(100.0 * per_class / max(drawn, 1), 1)})
    G.train()
    log = pd.DataFrame(log)
    display(log)
    return pd.concat(parts, ignore_index=True)[ref_df.columns], log

## Run

In [ ]:
def trial_key(c):
    return f"u{c['updates']}_b{c['batch']}_lr{c['lr']:.0e}_z{c['noise_dim']}"


CTX = prepare(df_train)
STEM_03 = f"ecwgan_E{ACTIVE}_{DATASET}_{trial_key(SETTING)}_seed{SEED}"
ck = torch.load(os.path.join(CHECKPOINT_DIR, f"ckpt_{STEM_03}.pt"),
                map_location=DEVICE, weights_only=False)
assert ck.get("kind") == "ecwgan" and ck.get("dataset") == DATASET and ck.get("active") == ACTIVE
assert trial_key(ck["cfg"]) == trial_key(SETTING) and ck.get("ct_dim") == CTX["ct"].dim
assert ck["updates_done"] >= SETTING["updates"], "notebook 03 did not finish training"

G = Generator(SETTING["noise_dim"], CTX["ct"].dim, CTX["cond_dim"]).to(DEVICE)
G.load_state_dict(ck["G"])
print(f"loaded {STEM_03} ({ck['updates_done']:,} updates)")

MODELS, NO_VAE = fit_class_vaes(df_train)

# Restore the RNG state saved at the end of notebook 03's training, so this pool
# starts from the same random draw as notebook 03's pool.
rng_restore(ck["rng"])
POOL, FILTER_LOG = build_pool_vae(G, CTX, SETTING, df_train, N_MAX, MODELS)
print("pool:", POOL.shape)

## Export

In [ ]:
STEM = f"ecwganVAE_{DATASET}_{trial_key(SETTING)}_seed{SEED}"
POOL_FILE = os.path.join(DATA_DIR, f"{STEM}_pool.csv")
LOG_FILE = os.path.join(DATA_DIR, f"{STEM}_filterlog.csv")
POOL.to_csv(POOL_FILE, index=False)
FILTER_LOG.to_csv(LOG_FILE, index=False)
print("written:", POOL_FILE, f"({len(POOL):,} rows)")

if IN_COLAB:
    files.download(POOL_FILE)
    files.download(LOG_FILE)